# Environment

In [ ]:
import os
os.chdir("..")

In [ ]:
import pandas as pd
from sklearn.metrics import classification_report

from src.pipeline.bert import PipelineBertBinary, PipelineBertMultilabel, BertPipelineConfig
from src.utils.path import DATA_PATH

from src.utils.report import multilabel_report

# Binary Classification

## Load Data

In [ ]:
df = pd.read_csv(f"{DATA_PATH}/hate-br/processed/hate-br.csv")

## Config

In [ ]:
config = BertPipelineConfig(
    model_path="./models/bert/binary",
    label_columns=["offensive"],
    train_args={
        "num_train_epochs": 5,
        "learning_rate": 2e-5,
        "per_device_train_batch_size": 32,
        "per_device_eval_batch_size": 64,
        "eval_strategy": "epoch",
        "save_strategy": "epoch",
        "logging_strategy": "epoch",
        "logging_steps": 1,
        "load_best_model_at_end": True,
        "logging_dir": "./logs",
        "report_to": "none",
        
        # Otimizações de Memória
        # "per_device_train_batch_size": 2,
        # "per_device_eval_batch_size": 4,
        # "gradient_accumulation_steps": 16,
        # "dataloader_pin_memory": False,
        # "bf16": True,
        # "fp16": False
    }
)

## Train

In [ ]:
pipeline = PipelineBertBinary(config)
train, val, test = pipeline.split(df)
trainer = pipeline.train(train, val, "text")

## Evaluate

In [ ]:
y_true, y_pred = pipeline.evaluate(trainer, test, "text")
print(classification_report(y_true, y_pred))

## Save

In [ ]:
pipeline.save(trainer)

# Multilabel Classification

In [ ]:
LABEL_COLUMNS = ["insult", "obscene", "ideology", "lgbtqphobia", "racism", "sexism", "xenophobia"]

## Load Data

In [ ]:
df = pd.read_csv(f"{DATA_PATH}/unified-hate/processed/unified-hate.csv")

## Config

In [ ]:
config = BertPipelineConfig(
    model_path="./models/bert/multilabel",
    label_columns=LABEL_COLUMNS,
    train_args={
        "num_train_epochs": 5,
        "learning_rate": 2e-5,
        "per_device_train_batch_size": 32,
        "per_device_eval_batch_size": 64,
        "eval_strategy": "epoch",
        "save_strategy": "epoch",
        "logging_strategy": "epoch",
        "logging_steps": 1,
        "load_best_model_at_end": True,
        "logging_dir": "./logs",
        "report_to": "none",
        
        # Otimizações de Memória
        # "per_device_train_batch_size": 2,
        # "per_device_eval_batch_size": 4,
        # "gradient_accumulation_steps": 16,
        # "dataloader_pin_memory": False,
        # "bf16": True,
        # "fp16": False
    }
)

## Train

In [ ]:
pipeline = PipelineBertMultilabel(config)
train, val, test = pipeline.split(df)
trainer = pipeline.train(train, val, "text")

## Evaluate

In [ ]:
y_true, y_pred = pipeline.evaluate(trainer, test, "text")
multilabel_report(y_true, y_pred, label_names=LABEL_COLUMNS)

## Save

In [ ]:
pipeline.save(trainer)